# 04 Resultaten inlezen en weergeven

In dit script worden de modelresultaten weergegeven en geplot.

In [1]:
import logging
import numpy as np
import pandas as pd
import geopandas as gpd
import hkvsobekpy
from pathlib import Path
import shutil
import matplotlib.pyplot as plt
from shapely.geometry import Point

In [2]:
%load_ext autoreload
%autoreload 2

### Inlezen meetlocaties en meetdata

Afvoermetingen

In [ ]:
def inlezen_csv_met_metadata(file_path: Path):
    meta = {}
    with open(file_path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    for line in lines:
        if not line.startswith("#"):
            continue
        text = line.strip("#").strip()
        if ":" in text:
            key, value = text.split(":", 1)
            meta[key.strip()] = value.strip().strip("; ")

    header_idx = next(i for i, line in enumerate(lines) if "Tijdstip (UTC);Waarde" in line)

    df = pd.read_csv(
        file_path,
        sep=";",
        skiprows=header_idx + 1,
        header=None,
        names=["Tijdstip (UTC)", "Waarde"],
        decimal=",",
        skipinitialspace=True
    )

    df["Tijdstip (UTC)"] = pd.to_datetime(df["Tijdstip (UTC)"], utc=True)
    df["time"] = df["Tijdstip (UTC)"].dt.tz_convert("Europe/Amsterdam").dt.tz_localize(None)
    df["Waarde"] = df["Waarde"].replace("---", np.nan).str.replace(",", ".").astype(float)
    df = df.set_index("time")[["Waarde"]]
    df.columns = [csv_file.stem]
    x = float(meta["Postitie X"].split(";")[0].strip("; (RD)"))
    y = float(meta["Postitie Y"].split(";")[0].strip("; (RD)"))

    return meta, Point(x,y), df

In [ ]:
dir_meetdata = Path("..\\..\\WRIJ_RR_Unpaved_methode_01_data\\IngekomenData\\20260520_meetdata_RR")
alle_metingen = pd.DataFrame()

afvoermeetlocaties = gpd.GeoDataFrame()
afvoermetingen = pd.DataFrame()

# afvoermetingen
dir_afvoermetingen = Path(dir_meetdata, "afvoermetingen_korte_naam")
csv_files = list(dir_afvoermetingen.glob("*.csv"))
for csv_file in csv_files:
    logging.info(f"Data van: {csv_file.stem}")
    meta, point, df = inlezen_csv_met_metadata(csv_file)
    afvoermeetlocaties = pd.concat([
        afvoermeetlocaties, 
        gpd.GeoDataFrame(
            [{"naam": csv_file.stem, "geometry": point}],
            geometry="geometry",
            crs=28992
        )
    ])
    afvoermetingen = pd.merge(afvoermetingen, df[csv_file.stem], how="outer", left_index=True, right_index=True)
afvoermetingen["2015-10-01":"2016-09-30"].plot()
afvoermeetlocaties.to_file(Path(dir_afvoermetingen, "afvoermeetlocaties.gpkg"))

In [ ]:
afvoermeetlocaties_gebieden = {
    1: {"in": [], "uit": ["Stuw Pelgrim (Waalse Water)"]},
    2: {"in": [], "uit": ["Overlaat Ulftseweg (Bergeslagbeek)"]},
    3: {"in": ["Debietmeting_Kotten_Vosseveldseweg"], "uit": ["Stuw Watermolen Berenschot"]}
}

### Selecteer welke modelresultaten (gebieden, scenario’s en periode) worden geanalyseerd

In [ ]:
# selectie_gebied = 0 # Oude IJssel
# selectie_gebied = 1 # West
# selectie_gebied = 2 # Centraal
# selectie_gebied = 3 # Oost

selectie_gebieden = [1, 2, 3]

scenarios = ["REF"]

# LONG RUN
# start_date = "2010-4-1"
# end_date = "2018-12-31"
# seizoenen = ["zomer", "winter"]
# date_range = pd.date_range(start_date, end_date, freq="2D")

# LONG TEST
start_date = "2015-07-1"
end_date = "2016-09-30"
seizoenen = ["zomer", "winter", "winter", "zomer"]
date_range = pd.date_range(start_date, end_date, freq="3MS")

# SMALL TEST
# start_date = "2012-4-1"
# end_date = "2012-4-12"
# seizoenen = ["zomer", "winter"]
# date_range = pd.date_range(start_date, end_date, freq="2D")

# path to the package containing the data
# dir_model_basis = Path("..\\..\\WRIJ_RR_Unpaved_methode_03_modellen\\oude_ijssel\\basis_test")
dir_model_basis = Path("..\\..\\WRIJ_RR_Unpaved_methode_03_modellen\\oude_ijssel\\basis_test_restart")

In [ ]:
simulations_total = pd.DataFrame()

for selectie_gebied in selectie_gebieden:
    for scenario in scenarios:

        simulaties = pd.DataFrame()
        simulaties["start_date"] = date_range
        simulaties["end_date"] = simulaties["start_date"].shift(-1)
        simulaties.loc[simulaties.index[-1],"end_date"] = pd.to_datetime(end_date)
        simulaties["seizoen"] = (seizoenen * 100)[:len(date_range)]
        simulaties["scenario"] = scenario
        simulaties["gebied"] = selectie_gebied
        simulaties["restart_in"] = [0] + [1] * len(simulaties.index[1:])

        simulaties["model_name"] = simulaties.apply(lambda x: f"rr_model__{pd.to_datetime(x.start_date).strftime('%Y_%m_%d')}__{pd.to_datetime(x.end_date).strftime('%Y_%m_%d')}", axis=1)
        simulations_total = pd.concat([simulations_total, simulaties])

In [ ]:
simulations_total

### ?? selecteer welk resultaat geplot moet worden (denk ik)

In [ ]:
total_link_flows = pd.DataFrame()

for selectie_gebied in selectie_gebieden:
    for scenario in scenarios:
        simulaties = simulations_total[(simulations_total["gebied"]==selectie_gebied) & (simulations_total["scenario"]==scenario)]
        link_flows = pd.DataFrame()
        for index, simulatie in simulaties.iterrows():
            print(str(simulatie.gebied) + " - " + simulatie.scenario + " - " + simulatie.model_name)

            dir_model = Path(dir_model_basis, f"gebied_{simulatie.gebied}", simulatie.scenario)
            unpaved_rr_file = "3blinks.his"

            path_unpaved_rr_file = Path(dir_model, simulatie.model_name, "rr", unpaved_rr_file)
            if not path_unpaved_rr_file.exists():
                print(f"File {path_unpaved_rr_file} does not exist. Skipping.")
                continue
            rr_his = hkvsobekpy.read_his.ReadMetadata(path_unpaved_rr_file)
            rr_results_link_flow = rr_his.DataFrame()['Link flow [m3/s]    ']
            link_flows = pd.concat([link_flows, rr_results_link_flow])
        
        total_link_flows[f"{selectie_gebied}_{scenario}"] = link_flows.sum(axis=1)

### Plot het resultaat

In [ ]:
start_date = "2015-07-01"
end_date = "2016-07-01"

selectie_scenario = ["REF"]
include_metingen = True

for selectie_gebied in selectie_gebieden:
    fig, axs = plt.subplots(2, 1, figsize=(12,10), sharex=True)
    total_link_flows_figure = total_link_flows[start_date:end_date]
    for scenario in selectie_scenario:

        total_link_flows_figure[[f"{selectie_gebied}_{scenario}"]].plot(ax=axs[1])

        if include_metingen:
            instroompunten = afvoermeetlocaties_gebieden[selectie_gebied]["in"]
            instroom = afvoermetingen[instroompunten].sum(axis=1)
            uitstroompunten = afvoermeetlocaties_gebieden[selectie_gebied]["uit"]
            uitstroom = afvoermetingen[uitstroompunten].sum(axis=1)
            if instroompunten:
                uitstroom.plot(
                    ax=axs[0], 
                    linestyle="None", 
                    marker="o",
                    markersize=1, 
                    color="red", 
                    label=f"{selectie_gebied} Uitstroom ({','.join(uitstroompunten)})"
                )
                instroom.plot(
                    ax=axs[0], 
                    linestyle="None", 
                    marker="o",
                    markersize=1, 
                    color="purple", 
                    label=f"{selectie_gebied} Instroom ({','.join(instroompunten)})"
                )
            for ax in axs:
                (uitstroom-instroom).plot(
                    ax=ax, 
                    linestyle="None", 
                    marker="o",
                    markersize=1, 
                    color="black", 
                    label=f"{selectie_gebied} Uitstroom ({','.join(uitstroompunten)}) - Instroom ({','.join(instroompunten)})"
                )

        ymin = 0
        ymax = 8
        # ymax = total_link_flows_figure[[f"{selectie_gebied}_{scenario}"]].max().max()*1.1
        axs[0].set_title(f"Totale afvoer | pilotgebied: {selectie_gebied} | Afvoermetingen")
        axs[1].set_title(f"Totale afvoer | pilotgebied: {selectie_gebied} | scenario: {scenario} vs Afvoermetingen")

        for ax in axs:
            ax.vlines(simulaties.start_date, ymin=ymin, ymax=ymax, color="lightgrey", linestyles="dashed")
            ax.set_ylim([ymin, ymax]);
            ax.set_xlim([start_date, end_date])

            # Titel en y-as label toevoegen
            ax.set_ylabel("Afvoer [m3/s]")
            ax.set_xlabel("")
            ax.grid()
            ax.legend()
    fig.savefig(Path(dir_model_basis, f"Resultaten_Gebied_{selectie_gebied}.png"))